In [31]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
import joblib

In [32]:
data = pd.read_csv('/content/dataset_laptop.csv')

print(df)

      cpu_usage  ram_usage    battery  temperature device_type    status
0     67.227856  12.200831  48.881862    44.135408      office    normal
1     48.807840  11.052144  81.593814          NaN      office   lagging
2     82.549517  11.559406  45.209614    50.792269      office       NaN
3           NaN   6.307530  23.593690    55.167093     student       NaN
4     80.804996   1.986335  80.093717    66.934921     student   lagging
...         ...        ...        ...          ...         ...       ...
2995        NaN  29.434094   1.578570    62.751216      office  overheat
2996  93.725804  17.868413  76.717186    53.673835     student   lagging
2997  63.493100   1.934684  46.606183    43.083879      gaming       NaN
2998  13.117076  24.198465  43.789536          NaN     student    normal
2999  49.003150   8.937702  92.765554    63.324048     student    normal

[3000 rows x 6 columns]


In [33]:
num_cols = ["cpu_usage", "ram_usage", "battery", "temperature"]
data[num_cols] = data[num_cols].fillna(data[num_cols].mean())

data["device_type"] = data["device_type"].fillna(data["device_type"].mode()[0])
data["status"] = data["status"].fillna(data["status"].mode()[0])


In [34]:
le_device = LabelEncoder()
data["device_type"] = le_device.fit_transform(data["device_type"])

le_status = LabelEncoder()
data["status"] = le_status.fit_transform(data["status"])


In [35]:
X = data.drop("status", axis=1)
y = data["status"]

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (2400, 5)
Test: (600, 5)


In [37]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [38]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

print("y shape after one-hot:", y_train.shape)

In [39]:
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')  # 3 classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [40]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.4125 - loss: 1.0830 - val_accuracy: 0.4542 - val_loss: 1.0680
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4443 - loss: 1.0712 - val_accuracy: 0.4521 - val_loss: 1.0678
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4422 - loss: 1.0676 - val_accuracy: 0.4542 - val_loss: 1.0658
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4453 - loss: 1.0650 - val_accuracy: 0.4604 - val_loss: 1.0694
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4437 - loss: 1.0641 - val_accuracy: 0.4563 - val_loss: 1.0665
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4427 - loss: 1.0635 - val_accuracy: 0.4479 - val_loss: 1.0666
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4458 - loss: 1.0619 - val_accuracy: 0.4521 - val_loss: 1.0687
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4458 - loss: 1.0598 - val_accuracy: 0.4542 - val_loss:

In [41]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4117 - loss: 1.1034  
Test Accuracy: 0.4116666615009308


In [42]:
sample = [[50, 16, 80, 60, 1]]  # cpu, ram, battery, temp, device_type

sample_scaled = scaler.transform(sample)

pred = model.predict(sample_scaled)
pred_class = np.argmax(pred)

print("Predicted class:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Predicted class: 0


In [ ]:
model.save("modellaptop.h5")
import joblib
joblib.dump(scaler, "scalerlabtop.pkl")